# 11 Small-class semantic segmentation + YOLO-large hybrid (baseline)

**New technical line.** All the box-based small-object attempts (two-stage, MedSAM, V15 NWD) closed at
the same wall: V6's tiny Caries boxes are *loose*, so anything that refines *after* a box inherits the
bad framing. **Semantic segmentation classifies pixels directly — it never goes through a box**, so it
is the one lever that can escape that wall on the small classes.

**Design (hybrid, chosen deliberately):**
- **LARGE classes (Abrasion / Crown / Filling) → keep YOLO (V6).** They carry the per-class-averaged
  mAP, they are near-saturated, and V6's boxes for them are already good — do not disturb them.
- **SMALL classes (all Caries) → semantic segmentation.** A `U-Net(resnet18, imagenet)` does multi-class
  per-pixel prediction over {background, caries...}; each connected component becomes one instance
  (polygon via contour, confidence = mean class probability over the component).
- **Merge** the two and score with the **same comparable Mask mAP as src/03/04/09**, so the delta vs V6
  is a true signal, not a metric artifact.

**Pre-registered reading (set before running):**
- **Headline = supported-small semseg AP vs V6 small AP** (Caries 1/2/3/5 only; Caries 4 n=4 / Caries 6
  n=5 are noise). Does boxless segmentation beat the loose-box detector on the small classes?
- Secondary = **hybrid aggregate (9 classes) vs V6 0.2099**. Remember: small classes are low-weight, so
  even a big small-class win moves the aggregate only modestly (the mAP-weight lesson).
- This is a **baseline** — fixed resize, plain CE, argmax→connected-components instances, mean-prob
  confidence. If the headline signal is positive, *then* refine (higher res, Dice loss, better instance
  split, TTA). If it is flat/negative even here, the line is likely not worth a full pipeline.

**Inputs (Kaggle):** the training dataset with `yolo_seg_train.yaml` (train+val images **and** labels)
and the **V6** detector (`version6...best.pt`). `segmentation_models_pytorch` + `ultralytics` (installed
if absent; this competition allows internet).

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys
for pkg, pipname in [("segmentation_models_pytorch", "segmentation-models-pytorch"),
                     ("ultralytics", "ultralytics")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])
print("deps ok")

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import cv2
import yaml
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from ultralytics import YOLO

cv2.setNumThreads(0)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Configuration

`LARGE_NAMES` are routed to YOLO; everything else (the caries classes) is segmented. The semantic model
sees a fixed `IMG_H x IMG_W` resize (panoramic X-rays are ~2:1, so a 1:2 input keeps the aspect roughly
right). Normalized polygon coordinates are invariant to the resize, so predicted instances map straight
back to full-image space for scoring — no per-image rescale needed on the semseg branch.

In [ ]:
# --- Hybrid routing ---
LARGE_NAMES = {"abrasion", "crown", "filling"}   # -> YOLO; all other (caries) classes -> semseg

# --- Semantic-seg model / training ---
ENCODER      = "resnet18"
ENCODER_WTS  = "imagenet"
IMG_H, IMG_W = 512, 1024      # fixed input (multiples of 32); ~matches panoramic aspect
EPOCHS       = 40
BATCH_SIZE   = 4
LR           = 1e-3
BG_WEIGHT    = 0.2            # CE weight on background (vs 1.0 per caries class) — severe class imbalance

# --- Instance extraction from the per-pixel map ---
MIN_COMPONENT_PX = 12         # drop connected components smaller than this (noise)
CAPTURE_CONF     = 0.001      # low: mask-AP needs the full PR curve (applies to the YOLO branch)

# --- Eval (identical to src/03/04/09) ---
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099
SUPPORTED_SMALL = {"caries 1","caries 2","caries 3","caries 5"}   # 4 (n=4) / 6 (n=5) are noise

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

## 3. Dataset + ground truth (train + val splits)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input (attach the training dataset).")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
num_classes = len(names)
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute(): yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

train_images = resolve_split(dcfg.get("train"))
val_images   = resolve_split(dcfg.get("val", dcfg.get("valid")))

def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try: cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError: return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

def load_objs(images_dir):
    objs = {}; ldir = labels_dir_for(images_dir)
    for ip in sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMG_EXTS):
        lp = ldir / (ip.stem + ".txt"); o = []
        if lp.exists():
            for line in lp.read_text().splitlines():
                pr = parse_seg_line(line)
                if pr is not None: o.append(pr)
        objs[str(ip)] = o
    return objs

train_objs = load_objs(train_images)
val_objs   = load_objs(val_images)

LARGE_IDX = [c for c in range(num_classes) if _norm_class_key(names[c]) in LARGE_NAMES]
SMALL_IDX = [c for c in range(num_classes) if c not in LARGE_IDX]   # caries -> semseg
small_to_sem = {c: i + 1 for i, c in enumerate(SMALL_IDX)}          # 0 = background
sem_to_small = {v: k for k, v in small_to_sem.items()}
N_SEM = len(SMALL_IDX) + 1
print("classes:", names)
print("LARGE (YOLO):", [names[c] for c in LARGE_IDX])
print("SMALL (semseg):", [names[c] for c in SMALL_IDX], "-> sem channels:", N_SEM)
print("train images:", len(train_objs), "| val images:", len(val_objs))

## 4. Semantic-segmentation dataset (rasterize small-class GT polygons)

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def rasterize_label(objs, out_h, out_w):
    # per-pixel class map over the SMALL classes; 0=bg, else small_to_sem. Later polys overwrite (ok for baseline).
    # NOTE: int32 (CV_32S), NOT int64 — cv2.fillPoly rejects int64 ("output array incompatible with cv::Mat").
    lbl = np.zeros((out_h, out_w), dtype=np.int32)
    for cls, poly in objs:
        if cls not in small_to_sem: continue
        pts = poly.copy(); pts[:, 0] *= out_w; pts[:, 1] *= out_h
        cv2.fillPoly(lbl, [pts.astype(np.int32)], int(small_to_sem[cls]))
    return lbl

def load_image_resized(ip):
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)

class SegDataset(torch.utils.data.Dataset):
    def __init__(self, objs_dict, augment=False):
        self.items = list(objs_dict.items()); self.augment = augment
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ip, objs = self.items[i]
        img = load_image_resized(ip)
        lbl = rasterize_label(objs, IMG_H, IMG_W)
        if self.augment and random.random() < 0.5:                 # hflip (label-safe; caries are not L/R-specific)
            img = img[:, ::-1, :].copy(); lbl = lbl[:, ::-1].copy()
        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        return x, torch.from_numpy(lbl).long()                     # CrossEntropyLoss target must be int64/long

train_loader = torch.utils.data.DataLoader(SegDataset(train_objs, augment=True),
                                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = torch.utils.data.DataLoader(SegDataset(val_objs, augment=False),
                                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("train/val datasets:", len(train_loader.dataset), len(val_loader.dataset))

## 5. Train the semantic segmenter (checkpoint by val foreground mIoU)

In [ ]:
model = smp.Unet(encoder_name=ENCODER, encoder_weights=ENCODER_WTS, in_channels=3, classes=N_SEM).to(DEVICE)
ce_weight = torch.tensor([BG_WEIGHT] + [1.0] * (N_SEM - 1), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=ce_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

@torch.no_grad()
def val_fg_miou():
    model.eval(); inter = np.zeros(N_SEM); union = np.zeros(N_SEM)
    for x, y in val_loader:
        pred = model(x.to(DEVICE)).argmax(1).cpu().numpy(); y = y.numpy()
        for cl in range(1, N_SEM):
            p = (pred == cl); g = (y == cl)
            inter[cl] += int((p & g).sum()); union[cl] += int((p | g).sum())
    ious = [inter[cl] / union[cl] for cl in range(1, N_SEM) if union[cl] > 0]
    return float(np.mean(ious)) if ious else 0.0

best_iou, best_state = -1.0, None
for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(x.to(DEVICE)), y.to(DEVICE))
        loss.backward(); optimizer.step(); tot += loss.item()
    viou = val_fg_miou()
    if viou > best_iou:
        best_iou = viou; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"ep {ep:3d}  train_loss {tot/len(train_loader):.4f}  val_fg_mIoU {viou:.4f}  best {best_iou:.4f}")
if best_state is not None: model.load_state_dict(best_state)
print(f"loaded best (val_fg_mIoU={best_iou:.4f})")

## 6. Semseg inference → small-class instances

Per-pixel softmax → argmax → per-class connected components. Each component (≥ `MIN_COMPONENT_PX`)
becomes one instance: polygon from the largest external contour, **confidence = mean class probability
over the component**, normalized polygon coords (resize-invariant).

In [ ]:
@torch.no_grad()
def predict_small_instances(ip):
    img = load_image_resized(ip)
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    x = ((x - IMAGENET_MEAN) / IMAGENET_STD).unsqueeze(0).to(DEVICE)
    model.eval()
    prob = torch.softmax(model(x), 1)[0].cpu().numpy()             # (N_SEM, H, W)
    argmax = prob.argmax(0)
    dets = []
    for cl in range(1, N_SEM):
        mask = (argmax == cl).astype(np.uint8)
        n, labels = cv2.connectedComponents(mask)
        for comp in range(1, n):
            cm = (labels == comp)
            if int(cm.sum()) < MIN_COMPONENT_PX: continue
            conf = float(prob[cl][cm].mean())
            cnts, _ = cv2.findContours(cm.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts: continue
            cnt = max(cnts, key=cv2.contourArea)
            if len(cnt) < 3: continue
            poly = cnt.reshape(-1, 2).astype(np.float64)
            poly[:, 0] /= IMG_W; poly[:, 1] /= IMG_H               # normalized [0,1]
            dets.append({"cls": int(sem_to_small[cl]), "conf": conf, "poly": poly})
    return dets

sem_dets = {ip: predict_small_instances(ip) for ip in tqdm(val_objs, desc="semseg val")}
print("semseg instances:", sum(len(v) for v in sem_dets.values()), "over", len(sem_dets), "images")

## 7. YOLO (V6) for the LARGE classes

In [ ]:
MANUAL_V6_PATH = None
def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

v6 = find_pt(["version6", "v6"], exclude=["stage2", "version9", "version10", "version13", "version14", "version15"])
V6_PATH = Path(MANUAL_V6_PATH) if MANUAL_V6_PATH else (v6[0] if v6 else None)
print("V6:", V6_PATH)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector found. Add version6_...best.pt or set MANUAL_V6_PATH.")
yolo = YOLO(str(V6_PATH))

def yolo_large_dets(ip):
    res = yolo.predict(source=ip, imgsz=768, conf=CAPTURE_CONF, iou=0.7, max_det=300,
                       device=(0 if DEVICE == "cuda" else "cpu"), task="segment", verbose=False, save=False)[0]
    out = []
    if res.masks is not None and res.boxes is not None and len(res.boxes) > 0:
        cls = res.boxes.cls.cpu().numpy().astype(int); conf = res.boxes.conf.cpu().numpy(); xyn = res.masks.xyn
        for i in range(min(len(cls), len(xyn))):
            if int(cls[i]) not in LARGE_IDX: continue                # large classes only
            poly = np.asarray(xyn[i], dtype=np.float64)
            if poly.ndim != 2 or len(poly) < 3: continue
            out.append({"cls": int(cls[i]), "conf": float(conf[i]), "poly": poly})
    return out

yolo_dets = {ip: yolo_large_dets(ip) for ip in tqdm(val_objs, desc="yolo large val")}
print("yolo large instances:", sum(len(v) for v in yolo_dets.values()))

## 8. Comparable Mask mAP — semseg-small vs V6, and the hybrid

Same matcher as src/03/04/09 (local-frame mask-IoU, greedy conf-ranked, 10 IoU thr, 101-pt AP).

In [ ]:
def poly_to_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8); pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3]); inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

n_gt = np.zeros(num_classes, dtype=int)
for o in val_objs.values():
    for c, _ in o: n_gt[c] += 1
wh = {}
for ip in val_objs:
    im = cv2.imread(ip); wh[ip] = (im.shape[1], im.shape[0])
gt_local = {ip: [(c, *poly_to_local_mask(p, *wh[ip])) for c, p in o] for ip, o in val_objs.items()}

def score(dets_by_image):
    records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip in val_objs:
        gts = gt_local[ip]
        pls = [(d["cls"], d["conf"], *poly_to_local_mask(d["poly"], *wh[ip])) for d in dets_by_image.get(ip, [])]
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order = sorted(range(len(pls)), key=lambda i: pls[i][1], reverse=True)
            used = [False] * len(gts)
            for pi in order:
                pc, sc, pbox, pm = pls[pi]; best, bg = 0.0, -1
                for gi, (gc, gbox, gm) in enumerate(gts):
                    if used[gi] or gc != pc: continue
                    v = iou_local(pbox, pm, gbox, gm)
                    if v > best: best, bg = v, gi
                tp = 1 if (bg >= 0 and best >= thr) else 0
                if tp: used[bg] = True
                records[(pc, ti)].append((sc, tp))
    pac = np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                    [r[1] for r in records[(c,ti)]], n_gt[c])
                             for ti in range(len(IOU_THRESHOLDS))])
    return pac

hybrid = {ip: yolo_dets[ip] + sem_dets[ip] for ip in val_objs}     # large->YOLO + small->semseg
ap_sem = score(sem_dets)                                           # semseg, small classes only
ap_hyb = score(hybrid)

def agg(pac, idxs):
    vals = [pac[c] for c in idxs if not np.isnan(pac[c])]
    return float(np.mean(vals)) if vals else float("nan")

print("per-class Mask AP — semseg (small) vs V6 reference:")
print(f"{'class':>14s}{'n_gt':>6s}{'semseg':>9s}{'V6_ref':>9s}{'delta':>9s}")
for c in SMALL_IDX:
    key = _norm_class_key(names[c]); ref = V6_REF_AP.get(key, float('nan'))
    tag = "" if key in SUPPORTED_SMALL else "  (noise)"
    print(f"{names[c]:>14s}{int(n_gt[c]):>6d}{ap_sem[c]:>9.3f}{ref:>9.3f}{ap_sem[c]-ref:>+9.3f}{tag}")

sup_idx = [c for c in SMALL_IDX if _norm_class_key(names[c]) in SUPPORTED_SMALL]
sem_sup = agg(ap_sem, sup_idx)
v6_sup  = float(np.mean([V6_REF_AP[_norm_class_key(names[c])] for c in sup_idx]))
print()
print(f"HEADLINE  supported-small (caries1/2/3/5): semseg {sem_sup:.4f}  vs V6 {v6_sup:.4f}  ({sem_sup-v6_sup:+.4f})")
print(f"HYBRID    aggregate (9 classes):           {agg(ap_hyb, range(num_classes)):.4f}  vs V6 {V6_REF_MAP:.4f}  ({agg(ap_hyb, range(num_classes))-V6_REF_MAP:+.4f})")

## 9. Save + how to read

- **`semseg_hybrid_baseline.csv`** = per-class semseg-small AP + hybrid AP + V6 reference.
- **`semseg_small_baseline.pt`** = the trained semantic segmenter (small classes), for follow-ups.

**Decision (pre-registered):**
- **Headline `supported-small semseg` clearly beats V6's supported-small AP** (beyond ~0.003) → the
  boxless route works on the small classes; pursue refinements (higher resolution, Dice/focal loss,
  better instance splitting, semseg TTA) and a test-set submission path.
- **Flat / negative even as a baseline** → boxless semseg does not rescue the small classes here either;
  the small-class headroom is genuinely capped (consistent with the mAP-weight reframing) → stop, keep
  the 2-model ensemble (LB 0.31753) as production.
- Either way the **hybrid aggregate** is the conservative number (small classes are low-weight); judge
  the *line* on the headline, judge a *submission* on the aggregate / the leaderboard.

In [ ]:
OUT = Path("/kaggle/working")
rows = []
for c in range(num_classes):
    rows.append({"class": names[c], "n_gt": int(n_gt[c]),
                 "semseg_small_AP": (None if np.isnan(ap_sem[c]) else float(ap_sem[c])),
                 "hybrid_AP": (None if np.isnan(ap_hyb[c]) else float(ap_hyb[c])),
                 "V6_ref": V6_REF_AP.get(_norm_class_key(names[c]))})
pd.DataFrame(rows).to_csv(OUT / "semseg_hybrid_baseline.csv", index=False)
torch.save(model.state_dict(), OUT / "semseg_small_baseline.pt")
print("saved /kaggle/working/semseg_hybrid_baseline.csv + semseg_small_baseline.pt")